In [1]:
from IPython import get_ipython
from IPython.display import display

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.5 MB/s eta 0:00:00


In [4]:
import os
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.svm import SVR
from skopt import gp_minimize
from skopt.space import Real, Categorical, Integer
from scipy.stats import zscore
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from scipy.stats import zscore

In [5]:
# Load dataset
features_train = pd.read_csv("/content/drive/MyDrive/Kuliah/Proposal/New/dengue_features_train.csv")
labels_train = pd.read_csv("/content/drive/MyDrive/Kuliah/Proposal/New/dengue_labels_train.csv")
features_test = pd.read_csv("/content/drive/MyDrive/Kuliah/Proposal/New/dengue_features_test.csv")

In [6]:
# Store 'week_start_date' separately for each city
week_start_date_train_sj = features_train.loc[features_train["city"] == "sj", 'week_start_date'].copy()
week_start_date_train_iq = features_train.loc[features_train["city"] == "iq", 'week_start_date'].copy()
week_start_date_test = features_test['week_start_date'].copy()

In [7]:
# Drop 'week_start_date' column from features dataframes
features_train.drop(columns=["week_start_date"], inplace=True)
features_test.drop(columns=["week_start_date"], inplace=True)

In [8]:
# Merge features and labels
train_data = features_train.merge(labels_train, on=["city", "year", "weekofyear"])

In [9]:
# Split dataset by city
sj_train = train_data[train_data["city"] == "sj"].drop(columns=["city"])
iq_train = train_data[train_data["city"] == "iq"].drop(columns=["city"])
features_test_sj = features_test[features_test["city"] == "sj"].drop(columns=['city'])
features_test_iq = features_test[features_test["city"] == "iq"].drop(columns=['city'])

Preprocessing

In [10]:
## Data Preprocessing

def preprocess_data(df_train, df_test, city_name):
    """Handles missing values, outlier detection (based on train), and standardization."""

    # Handle missing values (fill with median from training data)
    median_values = df_train.median(numeric_only=True)
    df_train.fillna(median_values, inplace=True)
    df_test.fillna(median_values, inplace=True) # Use training median for test data

    # Outlier Detection and Removal (based on training data)
    # Calculate Z-scores for training data (excluding 'total_cases' if it's the target)
    features_for_outlier_detection = df_train.drop(columns=['total_cases'] if 'total_cases' in df_train.columns else [], errors='ignore')
    z_scores_train = np.abs(zscore(features_for_outlier_detection))
    outliers_train_mask = (z_scores_train > 3).any(axis=1)
    outlier_rows_train = df_train.index[outliers_train_mask].tolist()
    print(f"Jumlah data dengan outlier di {city_name} train: {len(outlier_rows_train)}")

    # **Remove outliers from the training data**
    df_train_cleaned = df_train.drop(outlier_rows_train)
    print(f"Jumlah data train setelah outlier removal di {city_name}: {len(df_train_cleaned)}")


    # Separate features and target from the CLEANED training data
    if 'total_cases' in df_train_cleaned.columns:
        X_train = df_train_cleaned.drop(columns=['total_cases', 'year'], errors='ignore') # Drop year as it might not be a good feature for SVR/PCA
        y_train = df_train_cleaned['total_cases']
    else:
        X_train = df_train_cleaned.drop(columns=['year'], errors='ignore')
        y_train = None # Should not happen for train data

    X_test = df_test.drop(columns=['year'], errors='ignore') # Drop year from test data

    # Apply StandardScaler (fit on CLEANED train, transform CLEANED train and test)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test) # Use the same scaler fitted on train

    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

    # Apply PCA (fit on CLEANED train, transform CLEANED train and test)
    pca = PCA(n_components=12) # Choose number of components
    X_train_pca = pca.fit_transform(X_train_scaled_df)
    X_test_pca = pca.transform(X_test_scaled_df) # Use the same PCA fitted on train

    X_train_pca_df = pd.DataFrame(X_train_pca, index=X_train.index)
    X_test_pca_df = pd.DataFrame(X_test_pca, index=X_test.index)

    return X_train_pca_df, y_train, X_test_pca_df

In [11]:
# Preprocess data for San Juan and Iquitos
X_train_sj_pca, y_train_sj, X_test_sj_pca = preprocess_data(sj_train, features_test_sj.copy(), "San Juan")
X_train_iq_pca, y_train_iq, X_test_iq_pca = preprocess_data(iq_train, features_test_iq.copy(), "Iquitos")


print("\nProcessed data shapes:")
print("San Juan Train Features (PCA):", X_train_sj_pca.shape)
print("San Juan Train Target:", y_train_sj.shape)
print("San Juan Test Features (PCA):", X_test_sj_pca.shape)
print("Iquitos Train Features (PCA):", X_train_iq_pca.shape)
print("Iquitos Train Target:", y_train_iq.shape)
print("Iquitos Test Features (PCA):", X_test_iq_pca.shape)

Jumlah data dengan outlier di San Juan train: 87
Jumlah data train setelah outlier removal di San Juan: 849
Jumlah data dengan outlier di Iquitos train: 51
Jumlah data train setelah outlier removal di Iquitos: 469

Processed data shapes:
San Juan Train Features (PCA): (849, 12)
San Juan Train Target: (849,)
San Juan Test Features (PCA): (260, 12)
Iquitos Train Features (PCA): (469, 12)
Iquitos Train Target: (469,)
Iquitos Test Features (PCA): (156, 12)


Model Training and Evaluation

In [12]:
def train_and_evaluate_svr(X_train, y_train, X_test, y_test, city_name):
    """Trains and evaluates a basic SVR model."""
    svr_model = SVR()

    start_time = time.time()
    svr_model.fit(X_train, y_train)
    end_time = time.time()
    execution_time = end_time - start_time

    y_pred = svr_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n--- Basic SVR Results ({city_name}) ---")
    print(f"Mean Absolute Error: {mae}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Training Time: {execution_time:.4f} seconds")

    return svr_model # Return the trained model

In [13]:
def tune_and_evaluate_svr(X_train, y_train, X_test, y_test, city_name):
    """Performs GridSearchCV for SVR tuning and evaluates the best model."""
    param_grid_rbf = {
        'C': [0.01, 0.1, 1, 10, 100],
        'gamma': [0.001, 0.01, 0.1, 1],
        'kernel': ['rbf'],
        'epsilon': [0.001, 0.01, 0.1, 1, 10]
    }

    svr_model = SVR()
    grid_search = GridSearchCV(estimator=svr_model, param_grid=param_grid_rbf,
                              cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

    print(f"\nTuning SVR with GridSearchCV ({city_name})...")
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    end_time = time.time()
    execution_time = end_time - start_time

    best_svr_model = grid_search.best_estimator_
    y_pred = best_svr_model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n--- GridSearchCV SVR Results ({city_name}) ---")
    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Absolute Error: {mae}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Tuning and Evaluation Time: {execution_time:.4f} seconds")

    return best_svr_model # Return the best trained model

In [14]:
# Split training data for validation (for evaluation metrics)
X_train_sj_split, X_test_sj_split, y_train_sj_split, y_test_sj_split = train_test_split(
    X_train_sj_pca, y_train_sj, test_size=0.2, random_state=42)
X_train_iq_split, X_test_iq_split, y_train_iq_split, y_test_iq_split = train_test_split(
    X_train_iq_pca, y_train_iq, test_size=0.2, random_state=42)

In [15]:
# Train and evaluate basic SVR
basic_svr_sj = train_and_evaluate_svr(X_train_sj_split, y_train_sj_split, X_test_sj_split, y_test_sj_split, "San Juan (Basic)")
basic_svr_iq = train_and_evaluate_svr(X_train_iq_split, y_train_iq_split, X_test_iq_split, y_test_iq_split, "Iquitos (Basic)")


--- Basic SVR Results (San Juan (Basic)) ---
Mean Absolute Error: 21.419730976129905
Root Mean Squared Error: 47.93139601354305
Training Time: 0.0560 seconds

--- Basic SVR Results (Iquitos (Basic)) ---
Mean Absolute Error: 6.779105588388328
Root Mean Squared Error: 14.889522816160925
Training Time: 0.0173 seconds


In [16]:
# Tune and evaluate SVR with GridSearchCV
tuned_svr_sj = tune_and_evaluate_svr(X_train_sj_split, y_train_sj_split, X_test_sj_split, y_test_sj_split, "San Juan (Tuned)")
tuned_svr_iq = tune_and_evaluate_svr(X_train_iq_split, y_train_iq_split, X_test_iq_split, y_test_iq_split, "Iquitos (Tuned)")


Tuning SVR with GridSearchCV (San Juan (Tuned))...

--- GridSearchCV SVR Results (San Juan (Tuned)) ---
Best Hyperparameters: {'C': 100, 'epsilon': 1, 'gamma': 0.01, 'kernel': 'rbf'}
Mean Absolute Error: 20.062086188931804
Root Mean Squared Error: 45.81128595773383
Tuning and Evaluation Time: 32.7096 seconds

Tuning SVR with GridSearchCV (Iquitos (Tuned))...

--- GridSearchCV SVR Results (Iquitos (Tuned)) ---
Best Hyperparameters: {'C': 1, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf'}
Mean Absolute Error: 6.835315155343442
Root Mean Squared Error: 14.985738081110085
Tuning and Evaluation Time: 6.0244 seconds


# Task
Modify the code to use XGBoost instead of SVR for training and evaluating the dengue fever prediction model. Retain the existing steps for data loading, preprocessing, feature engineering, model training, hyperparameter tuning using GridSearchCV, prediction, and submission file generation.

## Update model training and evaluation functions

### Subtask:
Modify the existing functions `train_and_evaluate_svr` and `tune_and_evaluate_svr` to use XGBoost instead of SVR. This will involve changing the model object and potentially updating the hyperparameter grid for tuning.


**Reasoning**:
I need to modify the existing functions `train_and_evaluate_svr` and `tune_and_evaluate_svr` to use XGBoost. This involves changing the model and the hyperparameter grid for tuning. I will also rename the tuning function and update the print statements. I will group these changes into one code block as they are related to modifying the same functions.



In [21]:
import xgboost as xgb

def train_and_evaluate_xgboost(X_train, y_train, X_test, y_test, city_name):
    """Trains and evaluates a basic XGBoost model."""
    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

    start_time = time.time()
    xgb_model.fit(X_train, y_train)
    end_time = time.time()
    execution_time = end_time - start_time

    y_pred = xgb_model.predict(X_test)
    # Ensure predictions are non-negative integers
    y_pred = np.maximum(0, np.round(y_pred)).astype(int)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n--- Basic XGBoost Results ({city_name}) ---")
    print(f"Mean Absolute Error: {mae}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Training Time: {execution_time:.4f} seconds")

    return xgb_model # Return the trained model


def tune_and_evaluate_xgboost(X_train, y_train, X_test, y_test, city_name):
    """Performs GridSearchCV for XGBoost tuning and evaluates the best model."""
    param_grid_xgboost = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.9, 1.0],
    }

    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
    grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgboost,
                              cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

    print(f"\nTuning XGBoost with GridSearchCV ({city_name})...")
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    end_time = time.time()
    execution_time = end_time - start_time

    best_xgb_model = grid_search.best_estimator_
    y_pred = best_xgb_model.predict(X_test)
    # Ensure predictions are non-negative integers
    y_pred = np.maximum(0, np.round(y_pred)).astype(int)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n--- GridSearchCV XGBoost Results ({city_name}) ---")
    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Absolute Error: {mae}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Tuning and Evaluation Time: {execution_time:.4f} seconds")

    return best_xgb_model # Return the best trained model

## Add xgboost specific imports

### Subtask:
Add xgboost specific imports


**Reasoning**:
Add the import statement for the xgboost library to be used later in the notebook.



In [22]:
import xgboost as xgb

## Train and evaluate basic xgboost

### Subtask:
Use the modified basic training function to train and evaluate an XGBoost model.


**Reasoning**:
Call the modified basic training function for both cities using the PCA-transformed data and store the returned models.



In [23]:
basic_xgboost_sj = train_and_evaluate_xgboost(X_train_sj_split, y_train_sj_split, X_test_sj_split, y_test_sj_split, "San Juan (Basic XGBoost)")
basic_xgboost_iq = train_and_evaluate_xgboost(X_train_iq_split, y_train_iq_split, X_test_iq_split, y_test_iq_split, "Iquitos (Basic XGBoost)")


--- Basic XGBoost Results (San Juan (Basic XGBoost)) ---
Mean Absolute Error: 24.594117647058823
Root Mean Squared Error: 42.75009459913854
Training Time: 2.8790 seconds

--- Basic XGBoost Results (Iquitos (Basic XGBoost)) ---
Mean Absolute Error: 8.446808510638299
Root Mean Squared Error: 15.64703765043219
Training Time: 0.6175 seconds


## Tune and evaluate xgboost with gridsearchcv

### Subtask:
Use the modified tuning function to perform GridSearchCV for XGBoost and evaluate the best model.


**Reasoning**:
Use the modified tuning function to perform GridSearchCV for XGBoost and evaluate the best model for both cities as instructed.



In [24]:
tuned_xgboost_sj = tune_and_evaluate_xgboost(X_train_sj_split, y_train_sj_split, X_test_sj_split, y_test_sj_split, "San Juan (Tuned XGBoost)")
tuned_xgboost_iq = tune_and_evaluate_xgboost(X_train_iq_split, y_train_iq_split, X_test_iq_split, y_test_iq_split, "Iquitos (Tuned XGBoost)")


Tuning XGBoost with GridSearchCV (San Juan (Tuned XGBoost))...

--- GridSearchCV XGBoost Results (San Juan (Tuned XGBoost)) ---
Best Hyperparameters: {'colsample_bytree': 0.9, 'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 100, 'subsample': 0.7}
Mean Absolute Error: 25.070588235294117
Root Mean Squared Error: 42.915512072799366
Tuning and Evaluation Time: 580.9733 seconds

Tuning XGBoost with GridSearchCV (Iquitos (Tuned XGBoost))...

--- GridSearchCV XGBoost Results (Iquitos (Tuned XGBoost)) ---
Best Hyperparameters: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
Mean Absolute Error: 7.4361702127659575
Root Mean Squared Error: 14.559123775364469
Tuning and Evaluation Time: 493.4177 seconds
